# Codelist_Coverage — codelist-forward projection over cosmos-graph

Produces `cosmos-graph/interim/Codelist_Coverage.xlsx` with three sheets:

| Sheet | Grain | Source |
|---|---|---|
| `ReadMe` | — | provenance and column dictionary |
| `Codelists` | one row per codelist | `Variables.codelist_concept_id` aggregated; identity from `COSMoS_Graph_CT.Codelists`; governed term count from `COSMoS_Graph_CT.CodelistTerms` |
| `Codelist_Domain` | one row per (codelist, domain) | `Variables` joined to `DSS.domain`, aggregated by (codelist, domain) |

## Why

Three traversal directions exist over the COSMoS graph; only one is well-supported by `consumer-bases/DSS_View.xlsx` (DSS-forward). This artefact closes the codelist-forward direction.

Specifically, the `DSS_View` wide pivot keys on slot pins (`<remainder>_value` etc.). Bare codelist bindings — variables bound to a codelist with neither a pinned `assigned_term_value` nor a `value_list` restriction — drop out of the pivot. At 2026-Q1 there are **815 such bare-binding rows across 184 codelists** in the graph (e.g. AE.AELOC, AE.AESEV bound to LOC, AESEV codelists). This artefact preserves them.

## Inputs

| File | Track | Sheets used |
|---|---|---|
| `cosmos-graph/interim/COSMoS_Graph.xlsx` | cosmos-graph | DSS, Variables |
| `cosmos-graph/interim/COSMoS_Graph_CT.xlsx` | cosmos-graph | Codelists, CodelistTerms |

## Output

`cosmos-graph/interim/Codelist_Coverage.xlsx`

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

In [2]:
BASE_DIR = Path.cwd().parent          # cosmos-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

GRAPH_FILE = BASE_DIR / 'interim' / 'COSMoS_Graph.xlsx'
GRAPH_CT_FILE = BASE_DIR / 'interim' / 'COSMoS_Graph_CT.xlsx'

INTERIM_DIR = BASE_DIR / 'interim'
OUTPUT_FILE = INTERIM_DIR / 'Codelist_Coverage.xlsx'

for f, label in [(GRAPH_FILE, 'Graph'), (GRAPH_CT_FILE, 'Graph CT')]:
    if f.exists():
        print(f'  {label}: {f.relative_to(REPO_ROOT)}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')

  Graph: cosmos-graph/interim/COSMoS_Graph.xlsx
  Graph CT: cosmos-graph/interim/COSMoS_Graph_CT.xlsx
  Output: cosmos-graph/interim/Codelist_Coverage.xlsx


## 2. Load inputs

In [3]:
vars_g = pd.read_excel(GRAPH_FILE, sheet_name='Variables', dtype=str).fillna('')
dss_g = pd.read_excel(GRAPH_FILE, sheet_name='DSS', dtype=str).fillna('')

codelists_ct = pd.read_excel(GRAPH_CT_FILE, sheet_name='Codelists', dtype=str).fillna('')
codelist_terms = pd.read_excel(GRAPH_CT_FILE, sheet_name='CodelistTerms', dtype=str).fillna('')

print(f'Variables:      {len(vars_g):>6,} rows')
print(f'DSS:            {len(dss_g):>6,} rows')
print(f'Codelists:      {len(codelists_ct):>6,} rows  (CT-resolved)')
print(f'CodelistTerms:  {len(codelist_terms):>6,} rows  (one row per term across all codelists)')

Variables:      12,677 rows
DSS:             1,326 rows
Codelists:         291 rows  (CT-resolved)
CodelistTerms:  17,523 rows  (one row per term across all codelists)


## 3. Binding-mode masks

Three mutually-exclusive binding modes for any Variables row that carries a codelist:

- **pinned** — `assigned_term_value` is non-empty (the codelist is bound and one term is selected as the value)
- **value_list** — `value_list` is non-empty (the codelist is bound and a subset of terms is permitted)
- **bare** — neither pinned nor value_list (the codelist is bound; no further constraint at the variable level)

Pins and value_lists are mutually exclusive in COSMoS source. The three modes partition the codelist-bound Variables rows.

In [4]:
# Restrict to Variables rows that carry a codelist
vars_cl = vars_g[vars_g['codelist_concept_id'] != ''].copy()

vars_cl['mode'] = ''
vars_cl.loc[vars_cl['assigned_term_value'] != '', 'mode'] = 'pinned'
vars_cl.loc[(vars_cl['assigned_term_value'] == '') & (vars_cl['value_list'] != ''), 'mode'] = 'value_list'
vars_cl.loc[(vars_cl['assigned_term_value'] == '') & (vars_cl['value_list'] == ''), 'mode'] = 'bare'

# Sanity: every codelist-bound row must have a mode
unclassified = (vars_cl['mode'] == '').sum()
if unclassified:
    raise RuntimeError(f'{unclassified} codelist-bound Variables rows have no binding mode')

print(f'Codelist-bound Variables rows: {len(vars_cl):,}')
print(vars_cl['mode'].value_counts())

Codelist-bound Variables rows: 6,847
mode
pinned        4006
value_list    2026
bare           815
Name: count, dtype: int64


## 4. Codelists sheet — one row per codelist

Aggregations are at DSS grain: a single DSS can bind the same codelist on multiple variables with different modes (e.g. `--ORRESU` pinned to one unit, `--STRESU` restricted to a unit value_list, both bound to the same UNIT codelist). At codelist × DSS grain the modes are not mutually exclusive — `dss_count_binding` is the union, `dss_count_pinning + dss_count_value_list + dss_count_bare` is not.

Governed term count comes from `CodelistTerms` (one row per term across all codelists).

In [5]:
# Governed term count per codelist
term_counts = (
    codelist_terms.groupby('codelist_concept_id', sort=False).size()
    .rename('governed_term_count').reset_index()
)
print(f'Codelists with governed term counts: {len(term_counts):,}')

Codelists with governed term counts: 289


In [6]:
# Domain join — pull DSS.domain onto Variables-row context
ds_dom = dss_g[['ds_id', 'domain']].drop_duplicates()
vars_cl_dom = vars_cl.merge(ds_dom, on='ds_id', how='left').fillna('')
missing = (vars_cl_dom['domain'] == '').sum()
if missing:
    raise RuntimeError(f'{missing} codelist-bound Variables rows have no DSS.domain')

# Collapse to (codelist, ds_id, mode) — distinct combinations
per_dss_mode = (
    vars_cl_dom[['codelist_concept_id', 'ds_id', 'mode']]
    .drop_duplicates()
)
print(f'Distinct (codelist, ds_id, mode) tuples: {len(per_dss_mode):,}')

Distinct (codelist, ds_id, mode) tuples: 6,298


In [7]:
# Per-codelist DSS counts: distinct ds_id per codelist, overall and per mode
def _dss_count(df, mask):
    sub = df[mask][['codelist_concept_id', 'ds_id']].drop_duplicates()
    return sub.groupby('codelist_concept_id').size().rename('count').reset_index()

any_binding = _dss_count(per_dss_mode, per_dss_mode['mode'].notna()).rename(columns={'count': 'dss_count_binding'})
pinning     = _dss_count(per_dss_mode, per_dss_mode['mode'] == 'pinned').rename(columns={'count': 'dss_count_pinning'})
vlist       = _dss_count(per_dss_mode, per_dss_mode['mode'] == 'value_list').rename(columns={'count': 'dss_count_value_list'})
bare        = _dss_count(per_dss_mode, per_dss_mode['mode'] == 'bare').rename(columns={'count': 'dss_count_bare'})

# Domains (semicolon-joined list)
doms = (
    vars_cl_dom[['codelist_concept_id', 'domain']].drop_duplicates()
    .groupby('codelist_concept_id')['domain']
    .apply(lambda s: '; '.join(sorted(s)))
    .rename('domains_binding').reset_index()
)

print(f'Codelists with any binding:        {len(any_binding):,}')
print(f'Codelists with at least one pin:   {len(pinning):,}')
print(f'Codelists with at least one v_list: {len(vlist):,}')
print(f'Codelists with at least one bare:  {len(bare):,}')

Codelists with any binding:        291
Codelists with at least one pin:   109
Codelists with at least one v_list: 40
Codelists with at least one bare:  184


In [8]:
# Identity columns from CT-resolved Codelists
ident = codelists_ct[['codelist_concept_id', 'codelist_submission_value', 'codelist_name', 'codelist_extensible']].copy()

# Assemble Codelists sheet
codelists_out = (
    ident
    .merge(term_counts,  on='codelist_concept_id', how='left')
    .merge(any_binding,  on='codelist_concept_id', how='left')
    .merge(pinning,      on='codelist_concept_id', how='left')
    .merge(vlist,        on='codelist_concept_id', how='left')
    .merge(bare,         on='codelist_concept_id', how='left')
    .merge(doms,         on='codelist_concept_id', how='left')
)

# Fill counts: NaN → 0 for codelists with no rows in a given mode
for col in ['governed_term_count', 'dss_count_binding', 'dss_count_pinning',
            'dss_count_value_list', 'dss_count_bare']:
    codelists_out[col] = codelists_out[col].fillna(0).astype(int)
codelists_out['domains_binding'] = codelists_out['domains_binding'].fillna('')

codelists_out = codelists_out.sort_values('codelist_submission_value').reset_index(drop=True)
print(f'Codelists sheet: {len(codelists_out):,} rows x {len(codelists_out.columns)} cols')
codelists_out.head()

Codelists sheet: 291 rows x 10 cols


,codelist_concept_id,codelist_submission_value,codelist_name,codelist_extensible,governed_term_count,dss_count_binding,dss_count_pinning,dss_count_value_list,dss_count_bare,domains_binding
0,C189265,ACCPARTY,Accountable Party,Yes,8,7,0,0,7,BE
1,C66767,ACN,Action Taken with Study Treatment,No,8,2,0,0,2,AE
2,C100132,ADCTC,Alzheimer's Disease Assessment Scale - Cogniti...,No,111,16,16,0,0,FT
3,C100131,ADCTN,Alzheimer's Disease Assessment Scale - Cogniti...,No,111,16,16,0,0,FT
4,C66769,AESEV,Severity/Intensity Scale for Adverse Events,No,3,2,0,0,2,AE


## 5. Codelist_Domain sheet — one row per (codelist, domain)

Per-domain breakdown of the same DSS-grain counts. Long-format rather than a wide pivot (32 domains × 4 measures = 128 columns) so the artefact stays queryable.

In [9]:
# Per-(codelist, domain, ds_id, mode) — distinct
per_dom_mode = (
    vars_cl_dom[['codelist_concept_id', 'domain', 'ds_id', 'mode']]
    .drop_duplicates()
)

def _dom_count(df, mask):
    sub = df[mask][['codelist_concept_id', 'domain', 'ds_id']].drop_duplicates()
    return sub.groupby(['codelist_concept_id', 'domain']).size().rename('count').reset_index()

dom_any  = _dom_count(per_dom_mode, per_dom_mode['mode'].notna()).rename(columns={'count': 'dss_count_binding'})
dom_pin  = _dom_count(per_dom_mode, per_dom_mode['mode'] == 'pinned').rename(columns={'count': 'dss_count_pinning'})
dom_vl   = _dom_count(per_dom_mode, per_dom_mode['mode'] == 'value_list').rename(columns={'count': 'dss_count_value_list'})
dom_bare = _dom_count(per_dom_mode, per_dom_mode['mode'] == 'bare').rename(columns={'count': 'dss_count_bare'})

# Identity columns (codelist_submission_value alongside the concept_id for readability)
ident_dom = codelists_ct[['codelist_concept_id', 'codelist_submission_value']].copy()

codelist_domain_out = (
    dom_any
    .merge(ident_dom, on='codelist_concept_id', how='left')
    .merge(dom_pin,   on=['codelist_concept_id', 'domain'], how='left')
    .merge(dom_vl,    on=['codelist_concept_id', 'domain'], how='left')
    .merge(dom_bare,  on=['codelist_concept_id', 'domain'], how='left')
)

for col in ['dss_count_binding', 'dss_count_pinning', 'dss_count_value_list', 'dss_count_bare']:
    codelist_domain_out[col] = codelist_domain_out[col].fillna(0).astype(int)

# Final column order
codelist_domain_out = codelist_domain_out[[
    'codelist_concept_id', 'codelist_submission_value', 'domain',
    'dss_count_binding', 'dss_count_pinning', 'dss_count_value_list', 'dss_count_bare',
]].sort_values(['codelist_submission_value', 'domain']).reset_index(drop=True)

print(f'Codelist_Domain sheet: {len(codelist_domain_out):,} rows x {len(codelist_domain_out.columns)} cols')
codelist_domain_out.head()

Codelist_Domain sheet: 395 rows x 7 cols


,codelist_concept_id,codelist_submission_value,domain,dss_count_binding,dss_count_pinning,dss_count_value_list,dss_count_bare
0,C189265,ACCPARTY,BE,7,0,0,7
1,C66767,ACN,AE,2,0,0,2
2,C100132,ADCTC,FT,16,16,0,0
3,C100131,ADCTN,FT,16,16,0,0
4,C66769,AESEV,AE,2,0,0,2


## 6. Write workbook

Three sheets: `ReadMe`, `Codelists`, `Codelist_Domain`. Color convention follows the repo standard:

- **Yellow** headers — COSMoS-side identity (codelist concept id is grey since it is a key).
- **Grey** headers — keys, counts, aggregations.

In [10]:
# Styles
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')

GREEN_HEADER = PatternFill('solid', fgColor='548235')   # TESTCD / SDTM CT side
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # COSMoS side
GREY_HEADER = PatternFill('solid', fgColor='808080')    # keys, aggregation


def write_sheet(ws, df, header_fills, col_widths):
    cols = list(df.columns)
    for ci, name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=ci, value=name)
        cell.font = HEADER_FONT
        cell.fill = header_fills.get(name, GREY_HEADER)
        cell.alignment = WRAP
    for ri, (_, row) in enumerate(df.iterrows(), 2):
        for ci, name in enumerate(cols, 1):
            val = row[name]
            cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
            cell.font = DATA_FONT
            cell.alignment = WRAP
    for ci, name in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(name, 18)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'


print('Writer ready.')

Writer ready.


In [11]:
wb = Workbook()

# ── ReadMe sheet ──
ws_rm = wb.active
ws_rm.title = 'ReadMe'

readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

readme_lines = [
    ('Codelist_Coverage — codelist-forward projection over cosmos-graph', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    (f'Notebook: cosmos-graph/notebooks/40_codelist_coverage.ipynb', readme_font),
    (f'Inputs:', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph.xlsx (DSS, Variables)', readme_font),
    (f'  cosmos-graph/interim/COSMoS_Graph_CT.xlsx (Codelists, CodelistTerms)', readme_font),
    ('', None),
    ('SCOPE', section_font),
    ('All CDISC CT codelists referenced via Variables.codelist_concept_id', readme_font),
    ('across the COSMoS graph at 2026-Q1 — 291 codelists across 32 domains.', readme_font),
    ('', None),
    ('SCOPE DISCIPLINE', section_font),
    ('Mechanical aggregation over Variables and CodelistTerms. No editorial', readme_font),
    ('overlays. Counts are at DSS grain, not Variables-row grain — Variables', readme_font),
    ('detail belongs in DSS_Variables_View (consumer-bases, planned).', readme_font),
    ('', None),
    ('SHEETS', section_font),
    ('Codelists — one row per codelist.', readme_font),
    ('  codelist_concept_id      — NCIt C-code', readme_font),
    ('  codelist_submission_value — CDISC submission value', readme_font),
    ('  codelist_name            — CDISC codelist name', readme_font),
    ('  codelist_extensible      — extensibility flag', readme_font),
    ('  governed_term_count      — distinct terms in CodelistTerms', readme_font),
    ('  dss_count_binding        — distinct DSSs binding the codelist', readme_font),
    ('  dss_count_pinning        — distinct DSSs pinning at least one term', readme_font),
    ('  dss_count_value_list     — distinct DSSs restricting to a value_list', readme_font),
    ('  dss_count_bare           — distinct DSSs binding bare (no pin, no v_list)', readme_font),
    ('  domains_binding          — semicolon-joined list of domains touched', readme_font),
    ('Codelist_Domain — one row per (codelist, domain).', readme_font),
    ('  Same four DSS counts at codelist x domain grain.', readme_font),
    ('', None),
    ('BINDING MODES', section_font),
    ('Pinned, value_list-restricted, and bare are mutually exclusive at', readme_font),
    ('the Variables-row level. At codelist x DSS grain they are NOT mutually', readme_font),
    ('exclusive — a single DSS can bind the same codelist on multiple', readme_font),
    ('variables with different modes. So', readme_font),
    ('  dss_count_binding != dss_count_pinning + dss_count_value_list + dss_count_bare', readme_font),
    ('in general. The first is a union, the others are per-mode counts.', readme_font),
    ('', None),
    ('USE — closes the bare-codelist-fidelity gap', section_font),
    ('consumer-bases/DSS_View.xlsx Measurement_Specs pivots on slot pins.', readme_font),
    ('Variables that bind a codelist with neither pin nor value_list have', readme_font),
    ('no column in that wide pivot. At 2026-Q1 there are 815 such bare-', readme_font),
    ('binding rows across 184 codelists. This artefact preserves them.', readme_font),
    ('', None),
    ('STATUS', section_font),
    ('First codelist-forward projection at the cosmos-graph layer.', readme_font),
    ('Sources: COSMoS public exports + NCI EVS CT package 2026-03-27.', readme_font),
]

for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font

ws_rm.column_dimensions['A'].width = 100
print(f'ReadMe: {len(readme_lines)} lines')

ReadMe: 50 lines


In [12]:
# ── Codelists sheet ──
ws_cl = wb.create_sheet('Codelists')

CL_FILLS = {
    'codelist_concept_id':       GREY_HEADER,
    'codelist_submission_value': YELLOW_HEADER,
    'codelist_name':             YELLOW_HEADER,
    'codelist_extensible':       YELLOW_HEADER,
    'governed_term_count':       GREY_HEADER,
    'dss_count_binding':         GREY_HEADER,
    'dss_count_pinning':         GREY_HEADER,
    'dss_count_value_list':      GREY_HEADER,
    'dss_count_bare':            GREY_HEADER,
    'domains_binding':           GREY_HEADER,
}

CL_WIDTHS = {
    'codelist_concept_id':       14,
    'codelist_submission_value': 18,
    'codelist_name':             40,
    'codelist_extensible':       12,
    'governed_term_count':       12,
    'dss_count_binding':         12,
    'dss_count_pinning':         12,
    'dss_count_value_list':      14,
    'dss_count_bare':            12,
    'domains_binding':           40,
}

write_sheet(ws_cl, codelists_out, CL_FILLS, CL_WIDTHS)
print(f'Codelists: {len(codelists_out):,} rows x {len(codelists_out.columns)} cols')

Codelists: 291 rows x 10 cols


In [13]:
# ── Codelist_Domain sheet ──
ws_cd = wb.create_sheet('Codelist_Domain')

CD_FILLS = {
    'codelist_concept_id':       GREY_HEADER,
    'codelist_submission_value': YELLOW_HEADER,
    'domain':                    YELLOW_HEADER,
    'dss_count_binding':         GREY_HEADER,
    'dss_count_pinning':         GREY_HEADER,
    'dss_count_value_list':      GREY_HEADER,
    'dss_count_bare':            GREY_HEADER,
}

CD_WIDTHS = {
    'codelist_concept_id':       14,
    'codelist_submission_value': 18,
    'domain':                    8,
    'dss_count_binding':         12,
    'dss_count_pinning':         12,
    'dss_count_value_list':      14,
    'dss_count_bare':            12,
}

write_sheet(ws_cd, codelist_domain_out, CD_FILLS, CD_WIDTHS)
print(f'Codelist_Domain: {len(codelist_domain_out):,} rows x {len(codelist_domain_out.columns)} cols')

Codelist_Domain: 395 rows x 7 cols


In [14]:
wb.save(OUTPUT_FILE)
print(f'\nWritten: {OUTPUT_FILE}')
print(f'File size: {OUTPUT_FILE.stat().st_size / 1024:.0f} KB')


Written: /sessions/zealous-wonderful-einstein/mnt/cdisc-for-ai/cosmos-graph/interim/Codelist_Coverage.xlsx
File size: 37 KB


## 7. Summary

In [15]:
print('=== Codelist_Coverage summary ===')
print(f'Output: {OUTPUT_FILE.relative_to(REPO_ROOT)}')
print()
print(f'Codelists:        {len(codelists_out):>4,} rows x {len(codelists_out.columns):>2} cols')
print(f'Codelist_Domain:  {len(codelist_domain_out):>4,} rows x {len(codelist_domain_out.columns):>2} cols')
print()
print('Codelists with at least one row in each binding mode:')
print(f'  any binding:    {(codelists_out["dss_count_binding"]    > 0).sum():>4}  / {len(codelists_out)}')
print(f'  pinning:        {(codelists_out["dss_count_pinning"]    > 0).sum():>4}  / {len(codelists_out)}')
print(f'  value_list:     {(codelists_out["dss_count_value_list"] > 0).sum():>4}  / {len(codelists_out)}')
print(f'  bare:           {(codelists_out["dss_count_bare"]       > 0).sum():>4}  / {len(codelists_out)}')

=== Codelist_Coverage summary ===
Output: cosmos-graph/interim/Codelist_Coverage.xlsx

Codelists:         291 rows x 10 cols
Codelist_Domain:   395 rows x  7 cols

Codelists with at least one row in each binding mode:
  any binding:     291  / 291
  pinning:         109  / 291
  value_list:       40  / 291
  bare:            184  / 291
